# SkyAware Flight Delay Prediction — Notebook 4: Model Selection & Training

**AAI-540 MLOps | Group 4**

Steps:
1. Baseline models (Logistic Regression, Decision Tree) trained locally on SageMaker Studio
2. Primary model: **XGBoost** launched as a SageMaker Training Job (built-in container)
3. Training job metadata saved for downstream notebooks

## 1. Setup & Imports

In [1]:
import boto3
import pandas as pd
import numpy as np
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

import sagemaker
from sagemaker import get_execution_role
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight

region = boto3.Session().region_name
role = get_execution_role()
sagemaker_session = sagemaker.Session()
s3_client = boto3.client('s3', region_name=region)

PROCESSED_BUCKET = 'skyaware-processed-data'
MODEL_BUCKET = 'skyaware-model-artifacts'

# Ensure model bucket exists (EDA: 4 classes, imbalance ratio 0.398)
def create_bucket_if_missing(bucket_name):
    try:
        s3_client.head_bucket(Bucket=bucket_name)
        print(f'Bucket exists: s3://{bucket_name}')
    except Exception as e:
        error_code = e.response['Error']['Code'] if hasattr(e, 'response') else ''
        if error_code in ('404', 'NoSuchBucket'):
            try:
                if region == 'us-east-1':
                    s3_client.create_bucket(Bucket=bucket_name)
                else:
                    s3_client.create_bucket(
                        Bucket=bucket_name,
                        CreateBucketConfiguration={'LocationConstraint': region}
                    )
                print(f'Created bucket: s3://{bucket_name}')
            except Exception as create_err:
                print(f'Could not create {bucket_name}: {create_err}')
        else:
            print(f'Bucket check error for {bucket_name}: {e}')

create_bucket_if_missing(MODEL_BUCKET)

print(f'Region: {region}')
print(f'SageMaker SDK: {sagemaker.__version__}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Created bucket: s3://skyaware-model-artifacts
Region: us-east-1
SageMaker SDK: 2.224.4


## 2. Load Train & Validation Data

In [2]:
def load_split_from_s3(bucket, key):
    try:
        obj = s3_client.get_object(Bucket=bucket, Key=key)
        return pd.read_csv(obj['Body'])
    except Exception as e:
        print(f'Could not load {key}: {e}')
        return None

train_df = load_split_from_s3(PROCESSED_BUCKET, 'splits/train/train.csv')
val_df   = load_split_from_s3(PROCESSED_BUCKET, 'splits/val/val.csv')

if train_df is None or val_df is None:
    raise RuntimeError('Run Notebook 03 first to create the splits and upload to S3.')

TARGET_COL = 'delay_risk_label'
FEATURE_COLS = [c for c in train_df.columns if c != TARGET_COL]

X_train = train_df[FEATURE_COLS].values
y_train = train_df[TARGET_COL].values
X_val   = val_df[FEATURE_COLS].values
y_val   = val_df[TARGET_COL].values

print(f'Train: {X_train.shape}, Val: {X_val.shape}')
print(f'Features: {FEATURE_COLS}')
print(f'Classes: {np.unique(y_train)}')

Train: (140676, 30), Val: (70326, 30)
Features: ['year', 'month', 'month_sin', 'month_cos', 'quarter', 'is_winter', 'is_summer', 'is_peak_travel', 'is_covid_period', 'arr_flights', 'delay_rate', 'cancel_rate', 'divert_rate', 'avg_delay_per_flight', 'late_aircraft_rate', 'carrier_delay_rate', 'weather_delay_rate', 'nas_delay_rate', 'carrier_delay_pct', 'weather_delay_pct', 'nas_delay_pct', 'security_delay_pct', 'late_aircraft_pct', 'carrier_rolling_delay_3m', 'carrier_rolling_delay_12m', 'late_aircraft_roll_3m', 'airport_rolling_delay_3m', 'carrier_yoy_delay_change', 'carrier_enc', 'airport_enc']
Classes: [0. 1. 2. 3.]


## SageMaker Training Job: XGBoost

In [5]:
# XGBoost built-in container
xgb_image_uri = sagemaker.image_uris.retrieve(
    framework='xgboost',
    region=region,
    version='1.7-1'
)
print(f'XGBoost image URI: {xgb_image_uri}')

# Training job name with timestamp
job_name = f'skyaware-xgboost-{int(time.time())}'

xgb_estimator = Estimator(
    image_uri=xgb_image_uri,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{MODEL_BUCKET}/xgboost/',
    sagemaker_session=sagemaker_session,
    base_job_name='skyaware-xgboost',
    tags=[{'Key': 'Project', 'Value': 'SkyAware'}, {'Key': 'Version', 'Value': 'v1'}]
)

xgb_estimator.set_hyperparameters(
    objective='multi:softmax',
    num_class=4,
    max_depth=6,
    eta=0.1,
    num_round=200,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    eval_metric='merror',  # multi-class error rate
    early_stopping_rounds=20,
    seed=42
)

print('XGBoost estimator configured.')
print('Hyperparameters:', xgb_estimator.hyperparameters())

XGBoost image URI: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1
XGBoost estimator configured.
Hyperparameters: {'objective': 'multi:softmax', 'num_class': 4, 'max_depth': 6, 'eta': 0.1, 'num_round': 200, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 3, 'eval_metric': 'merror', 'early_stopping_rounds': 20, 'seed': 42}


In [6]:
# XGBoost built-in requires: target as FIRST column, NO header
# Files train_xgb.csv and val_xgb.csv were prepared in NB03

train_input = TrainingInput(
    f's3://{PROCESSED_BUCKET}/splits/train/train_xgb.csv',
    content_type='text/csv'
)
val_input = TrainingInput(
    f's3://{PROCESSED_BUCKET}/splits/val/val_xgb.csv',
    content_type='text/csv'
)

print('Launching XGBoost training job...')
print(f'Train data: s3://{PROCESSED_BUCKET}/splits/train/train_xgb.csv')
print(f'Val data:   s3://{PROCESSED_BUCKET}/splits/val/val_xgb.csv')
print(f'Output:     s3://{MODEL_BUCKET}/xgboost/')

xgb_estimator.fit(
    inputs={'train': train_input, 'validation': val_input},
    job_name=job_name,
    logs=True
)

training_job_name = xgb_estimator.latest_training_job.name
model_artifact = xgb_estimator.model_data

print(f'\nTraining job completed!')
print(f'Job name:       {training_job_name}')
print(f'Model artifact: {model_artifact}')

INFO:sagemaker:Creating training-job with name: skyaware-xgboost-1781965796


Launching XGBoost training job...
Train data: s3://skyaware-processed-data/splits/train/train_xgb.csv
Val data:   s3://skyaware-processed-data/splits/val/val_xgb.csv
Output:     s3://skyaware-model-artifacts/xgboost/


2026-06-20 14:30:16 Starting - Starting the training job.

.

.

...

.

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-06-20 14:32:20.110 ip-10-0-246-116.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-06-20 14:32:20.183 ip-10-0-246-116.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-06-20:14:32:20:INFO] Imported framework sagemaker_xgboost_container.training
[2026-06-20:14:32:20:INFO] Failed to parse hyperparameter eval_metric value merror to Json.
Returning the value itself
[2026-06-20:14:32:20:INFO] Failed to parse hyperparameter objective value multi:softmax to Json.
Returning the value itself
[2026-06-20:14:32:20:INFO] No GPUs detected (normal if no gpus installed)
[2026-06-20:14:32:20:INFO] Ru


2026-06-20 14:32:38 Uploading - Uploading generated training model


2026-06-20 14:32:51 Completed - Training job completed


Training seconds: 114
Billable seconds: 114

Training job completed!
Job name:       skyaware-xgboost-1781965796
Model artifact: s3://skyaware-model-artifacts/xgboost/skyaware-xgboost-1781965796/output/model.tar.gz


## Save Training Job Metadata

In [8]:
training_info = {
    'training_job_name': training_job_name,
    'model_artifact_s3': model_artifact,
    'xgb_image_uri': xgb_image_uri,
    'baseline_lr_f1': float(lr_f1),
    'baseline_dt_f1': float(dt_f1),
    'feature_columns': FEATURE_COLS,
    'target_column': TARGET_COL,
    'num_classes': 4,
    'label_names': {0: 'on-time', 1: 'minor-risk', 2: 'major-risk', 3: 'high-risk'}
}

# Save locally
with open('training_job_info.json', 'w') as f:
    json.dump(training_info, f, indent=2)
print('Saved training_job_info.json locally')

# Upload to S3
try:
    s3_client.upload_file('training_job_info.json', MODEL_BUCKET, 'metadata/training_job_info.json')
    print(f'Saved to s3://{MODEL_BUCKET}/metadata/training_job_info.json')
except Exception as e:
    print(f'S3 upload note: {e}')

print('\nTraining job info:')
for k, v in training_info.items():
    if isinstance(v, list):
        print(f'  {k}: [{len(v)} items]')
    else:
        print(f'  {k}: {v}')

Saved training_job_info.json locally
Saved to s3://skyaware-model-artifacts/metadata/training_job_info.json

Training job info:
  training_job_name: skyaware-xgboost-1781965796
  model_artifact_s3: s3://skyaware-model-artifacts/xgboost/skyaware-xgboost-1781965796/output/model.tar.gz
  xgb_image_uri: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1
  baseline_lr_f1: 0.5793153099720715
  baseline_dt_f1: 1.0
  feature_columns: [30 items]
  target_column: delay_risk_label
  num_classes: 4
  label_names: {0: 'on-time', 1: 'minor-risk', 2: 'major-risk', 3: 'high-risk'}
